# Activation Steering

vLLM-Hook is an extensible framework that aims to allow selective access to model internals during inference.
In this notebook, we demonstrate how vLLM-Hook enables **Activation Steering** for controlled generation.

**Paper**: [Improving Instruction-Following in Language Models through Activation Steering](https://arxiv.org/abs/2410.12877).<br />
**Authors**: Alessandro Stolfo, Vidhisha Balachandran, Safoora Yousefi, Eric Horvitz, Besmira Nushi <br />
**"TL;DR"**: Activation steering allows you to bias the model's behavior by nudging internal activations in specific directions. In this paper, authors focus on instruction following capability and compute the steering vectors as the difference in activations between inputs with and without instructions. 

### Installation

If running this from a new environment, please use the cell below to install `vllm_hook_plugins`. Update the path/command to match your environment.<br />
The following block is not necessary if running this notebook from an environment where the package has already been installed.

In [1]:
from pathlib import Path
import sys

# vllm_hooks/notebooks/
NOTEBOOK_DIR = Path.cwd()
REPO_ROOT = NOTEBOOK_DIR.parent.parent

PKG_DIR = REPO_ROOT/"vllm_hook_plugins"
REQ_FILE = REPO_ROOT/"requirement.txt"

print("Notebook dir:", NOTEBOOK_DIR)
print("Repo root   :", REPO_ROOT)
print("Package dir :", PKG_DIR)
print("Req file    :", REQ_FILE)

%pip install -e "{PKG_DIR}"

if REQ_FILE.exists():
    %pip install -r "{REQ_FILE}"
else:
    print("⚠️ requirements.txt not found at", REQ_FILE)


Notebook dir: /Users/timothyburley/opensource/vLLM-Hook/notebooks/metal
Repo root   : /Users/timothyburley/opensource/vLLM-Hook
Package dir : /Users/timothyburley/opensource/vLLM-Hook/vllm_hook_plugins
Req file    : /Users/timothyburley/opensource/vLLM-Hook/requirement.txt
Obtaining file:///Users/timothyburley/opensource/vLLM-Hook/vllm_hook_plugins
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for vllm-hook-plugins (pyproject.toml) ... done
  Created wheel for vllm-hook-plugins: filename=vllm_hook_plugins-0.2.0-0.editable-py3-none-any.whl size=3147 sha256=7a70eb19db196e549e8d4488a41acf599f9d34d02027b3ea0ea527176d4ec5c3
  Stored in directory: /private/var/folders/dn/99pbhj4d48n4r8rg_hvqtglr0000gn/T/pip-ephem-wheel-cache-2xddgc1u/wheels/91/fa/cf/bacb8fa72ad781d6b97e1ba762fa3be0ed4d9aa39201e4b56d
Successfully 

### Importing the Hook-Enabled LLM
The plugin provides its own LLM wrapper that behaves like vllm.LLM (`from vllm import LLM`) but adds support for hooks and instrumentation.
We import it here:

In [2]:
from vllm_hook_plugins.metal import HookLLMMetal

INFO 06-07 17:09:33 [__init__.py:44] Available plugins for group vllm.platform_plugins:
INFO 06-07 17:09:33 [__init__.py:46] - metal -> vllm_metal:register
INFO 06-07 17:09:33 [__init__.py:49] All plugins in this group will be loaded. Set `VLLM_PLUGINS` to control which plugins to load.
INFO 06-07 17:09:37 [__init__.py:238] Platform plugin metal is activated
INFO 06-07 17:09:37 [importing.py:69] Triton not installed or not compatible; certain GPU-related functions will not be available.


### Environment & multiprocessing setup

In [3]:
import gc
import os
import multiprocessing as mp
import torch
from pathlib import Path
from vllm import SamplingParams
mp.set_start_method("spawn", force=True)
os.environ["VLLM_WORKER_MULTIPROC_METHOD"] = "spawn"

# Steering vector paths in model_configs/.../*.json are relative to the repo
# root, so chdir there before constructing HookLLM.
os.chdir(Path.cwd().parent.parent if Path.cwd().name == "metal" else Path.cwd())

### Initialize `HookLLMMetal`
Before we create the LLM instance, we need to specify the model and data type:

In [4]:
cache_dir = './cache'  # Specify cache dir
model = 'mlx-community/Phi-3.5-mini-instruct-4bit'
hf_config_path = model
# Use the same repo for tokenizer/config so vLLM can resolve the bundled files.

dtype = 'auto'


We also need to provide a config file that specifies how activations are steered (e.g., which layers to intervene on, which token to intervene, what direction vectors to apply, etc.).<br />
In the following example, we apply activation steering at the 15th layer, apply the steering at all positions (as opposed to only at the start of the decoding process), and along the direction given in `vector_path`:

In [5]:
import json

json_path = Path("model_configs/activation_steer/Phi-3-mini-4k-instruct.json")  # reused until a Phi-3.5-specific steering config is added

with open(json_path, "r") as f:
    config = json.load(f)

# print(config)

In [6]:
try:
    import mlx.core as mx
except Exception:
    mx = None

def _clear_model_caches():
    gc.collect()
    if mx is not None and hasattr(mx, "clear_cache"):
        mx.clear_cache()
        
_clear_model_caches()

Inside `steer_hook_act` we defined the activation steering behavior during model inference.
Now, we initialize the llm:

In [ ]:
llm = HookLLMMetal(
    model=model,
    tokenizer=hf_config_path,
    worker_name="steer_hook_act",
    config_file=json_path,
    download_dir=cache_dir,
    hf_config_path=hf_config_path,
    gpu_memory_utilization=0.2,
    trust_remote_code=True,
    dtype=dtype,
    enable_prefix_caching=False,
    enable_hook=True,

    
    # May run into an MLX limit if unset.
    max_model_len=2048,
    max_num_seqs=2,
    max_num_batched_tokens=512,
)

HookLLMMetal worker=steer_hook_act hooks_enabled=True
INFO 06-07 17:09:39 [utils.py:278] non-default args: {'tokenizer': 'mlx-community/Phi-3.5-mini-instruct-4bit', 'hf_config_path': 'mlx-community/Phi-3.5-mini-instruct-4bit', 'trust_remote_code': True, 'download_dir': './cache', 'enable_prefix_caching': False, 'gpu_memory_utilization': 0.2, 'disable_log_stats': True, 'enforce_eager': True, 'model': 'mlx-community/Phi-3.5-mini-instruct-4bit'}
WARNING 06-07 17:09:39 [envs.py:2057] Unknown vLLM environment variable detected: VLLM_HOOK_WORKER
WARNING 06-07 17:09:39 [arg_utils.py:1552] The global random seed is set to 0. Since VLLM_ENABLE_V1_MULTIPROCESSING is set to False, this may affect the random state of the Python process that launched vLLM.


[transformers] This model config has set a `rope_parameters['original_max_position_embeddings']` field, to be used together with `max_position_embeddings` to determine a scaling factor. Please set the `factor` field of `rope_parameters`with this ratio instead -- we recommend the use of this field over `original_max_position_embeddings`, as it is compatible with most model architectures.


INFO 06-07 17:09:40 [model.py:617] Resolved architecture: Phi3ForCausalLM
INFO 06-07 17:09:40 [model.py:1752] Using max model len 4096
INFO 06-07 17:09:40 [scheduler.py:239] Chunked prefill is enabled with max_num_batched_tokens=8192.
INFO 06-07 17:09:40 [vllm.py:977] Asynchronous scheduling is enabled.
WARNING 06-07 17:09:40 [vllm.py:1033] Enforce eager set, disabling torch.compile and CUDAGraphs. This is equivalent to setting -cc.mode=none -cc.cudagraph_mode=none
WARNING 06-07 17:09:40 [vllm.py:1058] Inductor compilation was disabled by user settings, optimizations settings that are only active during inductor compilation will be ignored.
INFO 06-07 17:09:40 [kernel.py:270] Final IR op priority after setting platform defaults: IrOpPriorityConfig(rms_norm=['native'], fused_add_rms_norm=['native'])
INFO 06-07 17:09:40 [platform.py:286] Metal: disabled chunked prefill (non-paged path), max_num_batched_tokens=8192
INFO 06-07 17:09:41 [platform.py:324] Metal memory: 34.4GB total, 13.1GB a

mx.metal.device_info is deprecated and will be removed in a future version. Use mx.device_info instead.


INFO 06-07 17:09:43 [model_lifecycle.py:126] Loading model: mlx-community/Phi-3.5-mini-instruct-4bit (VLM: False)


Fetching 11 files:   0%|          | 0/11 [00:00<?, ?it/s]

INFO 06-07 17:09:43 [model_lifecycle.py:187] Model loaded in 0.69s: mlx-community/Phi-3.5-mini-instruct-4bit
INFO 06-07 17:09:43 [cache_policy.py:680] MLX path: reporting 1.61 GB for scheduler admission control (one max-length sequence, max_model_len=4096)
INFO 06-07 17:09:43 [kv_cache_utils.py:1733] GPU KV cache size: 4,096 tokens
INFO 06-07 17:09:43 [kv_cache_utils.py:1734] Maximum concurrency for 4,096 tokens per request: 1.00x
INFO 06-07 17:09:43 [cache_policy.py:297] KV cache config received: 256 blocks (MLX manages cache internally)
INFO 06-07 17:09:43 [model_runner.py:649] Warming up model...
INFO 06-07 17:09:43 [model_runner.py:655] Model warm-up complete
INFO 06-07 17:09:43 [core.py:302] init engine (profile, create kv cache, warmup model) took 0.10 s (compilation: 0.09 s)


### Test case
In the following, we show a test case and compare generations **with** and **without** activation steering.

**Note**: Users should swap the example configs with their own to show desirable performance. The following is for pipeline illustration only.

In [8]:
test_cases = [
    "Write a dialogue between two people, one is dressed up in a ball gown and the other is dressed down in sweats. The two are going to a nightly event. Your answer must contain exactly 3 bullet points in the markdown format (use \"* \" to indicate each bullet) such as:\n* This is the first point.\n* This is the second point.",
    "What is the difference between the 13 colonies and the other British colonies in North America? Your answer must contain exactly 6 bullet point in Markdown using the following format:\n* Bullet point one.\n* Bullet point two.\n...\n* Bullet point fix."
]

Before we start, we define the sampling parameters:

**Note**: token 32007 is phi-specific, refer to the original huggingface implementation for details https://github.com/microsoft/llm-steer-instruct/blob/main/utils/generation_utils.py.

- Option 1: we can use a uniform sampling parameters as follows

In [9]:
# sampling_params = SamplingParams(
#     temperature=0.0,                       
#     max_tokens=2048,
#     stop_token_ids=[llm.tokenizer.eos_token_id, 32007],  
# )

- Option 2: pass a list of `SamplingParams` aligned with the prompts, matching the non-Metal notebook API. The current Metal steer worker uses the JSON steering config for the hooked engine; the list shape is still useful here because the baseline call can reuse the same sampling parameters after `use_hook=False` strips hook-only metadata. Unlike PyTorch forward hooks, Metal uses MLX module wrapper replacement, so it wraps only the configured target layer by default; set `VLLM_HOOK_METAL_STEER_ALL_LAYERS=1` only when debugging all-layer wrapper parity.

In [10]:
base_steer = config["steering"]  # full default config from the JSON
sampling_params_list = [
    SamplingParams(
        temperature=0.0,
        max_tokens=32,
        stop_token_ids=[llm.tokenizer.eos_token_id, 32007],
    ),
    SamplingParams(
        temperature=0.0,
        max_tokens=32,
        stop_token_ids=[llm.tokenizer.eos_token_id, 32007],
        extra_args={"steer": {**base_steer, "method": "add_vector", "coefficient": 10}},
    ),
]

Next we:
1. Apply chat template on each test case,
2. Generate all prompts in one batched call instead of rebuilding the Metal hook engine once per prompt,
3. Reset the prefix cache to ensure the baseline generation does not reuse steered cache,
4. Generate again with `use_hook=False` to obtain the baseline output.

In [11]:
examples = [
    llm.tokenizer.apply_chat_template(
        [{"role": "user", "content": case}], add_generation_prompt=True, tokenize=False
    )
    for case in test_cases
]

outputs = llm.generate(examples, sampling_params_list)

# Keep the baseline independent from any steered KV/prefix state.
llm.llm_engine.reset_prefix_cache()
_clear_model_caches()

outputs_original = llm.generate(examples, sampling_params_list, use_hook=False)
_clear_model_caches()

Metal steer: preparing hook run
Metal steer: host before hook setup rss=2.74 GB available=9.97 GB total=32.00 GB
Releasing base engine before Metal steer-hook capture.
Metal steer: building hook engine
Metal steer: host before hook engine build rss=2.74 GB available=9.98 GB total=32.00 GB
INFO 06-07 17:09:44 [utils.py:278] non-default args: {'tokenizer': 'mlx-community/Phi-3.5-mini-instruct-4bit', 'hf_config_path': 'mlx-community/Phi-3.5-mini-instruct-4bit', 'trust_remote_code': True, 'download_dir': './cache', 'enable_prefix_caching': False, 'gpu_memory_utilization': 0.2, 'disable_log_stats': True, 'enforce_eager': True, 'model': 'mlx-community/Phi-3.5-mini-instruct-4bit'}
WARNING 06-07 17:09:44 [envs.py:2057] Unknown vLLM environment variable detected: VLLM_HOOK_WORKER
WARNING 06-07 17:09:44 [arg_utils.py:1552] The global random seed is set to 0. Since VLLM_ENABLE_V1_MULTIPROCESSING is set to False, this may affect the random state of the Python process that launched vLLM.
INFO 06-07

Rendering prompts:   0%|          | 0/1 [00:00<?, ?it/s]

Processed prompts: 100%|██████████| 1/1 [00:01<00:00,  1.32s/it, est. speed input: 62.42 toks/s, output: 24.36 toks/s]

Metal steer: hooked generation complete
Metal steer: cleaning up hook run
Metal steer: host after hook exit rss=2.76 GB available=9.89 GB total=32.00 GB
Metal steer: preparing hook run
Metal steer: host before hook setup rss=2.76 GB available=9.89 GB total=32.00 GB
Releasing base engine before Metal steer-hook capture.
Metal steer: building hook engine
Metal steer: host before hook engine build rss=2.76 GB available=9.89 GB total=32.00 GB
INFO 06-07 17:09:47 [utils.py:278] non-default args: {'tokenizer': 'mlx-community/Phi-3.5-mini-instruct-4bit', 'hf_config_path': 'mlx-community/Phi-3.5-mini-instruct-4bit', 'trust_remote_code': True, 'download_dir': './cache', 'enable_prefix_caching': False, 'gpu_memory_utilization': 0.2, 'disable_log_stats': True, 'enforce_eager': True, 'model': 'mlx-community/Phi-3.5-mini-instruct-4bit'}
WARNING 06-07 17:09:47 [envs.py:2057] Unknown vLLM environment variable detected: VLLM_HOOK_WORKER
WARNING 06-07 17:09:47 [arg_utils.py:1552] The global random seed

INFO 06-07 17:09:48 [model.py:617] Resolved architecture: Phi3ForCausalLM
INFO 06-07 17:09:48 [model.py:1752] Using max model len 4096
WARNING 06-07 17:09:48 [vllm.py:1033] Enforce eager set, disabling torch.compile and CUDAGraphs. This is equivalent to setting -cc.mode=none -cc.cudagraph_mode=none
WARNING 06-07 17:09:48 [vllm.py:1058] Inductor compilation was disabled by user settings, optimizations settings that are only active during inductor compilation will be ignored.
INFO 06-07 17:09:48 [kernel.py:270] Final IR op priority after setting platform defaults: IrOpPriorityConfig(rms_norm=['native'], fused_add_rms_norm=['native'])
INFO 06-07 17:09:48 [platform.py:286] Metal: disabled chunked prefill (non-paged path), max_num_batched_tokens=8192
INFO 06-07 17:09:48 [platform.py:324] Metal memory: 34.4GB total, 10.7GB available
INFO 06-07 17:09:48 [core.py:112] Initializing a V1 LLM engine (v0.22.0) with config: model='mlx-community/Phi-3.5-mini-instruct-4bit', speculative_config=None, 

...

Processed prompts: 100%|██████████| 1/1 [00:01<00:00,  1.30s/it, est. speed input: 49.37 toks/s, output: 24.68 toks/s]

Metal steer: hooked generation complete
Metal steer: cleaning up hook run
Metal steer: host after hook exit rss=2.80 GB available=9.70 GB total=32.00 GB
INFO 06-07 17:09:50 [utils.py:278] non-default args: {'tokenizer': 'mlx-community/Phi-3.5-mini-instruct-4bit', 'hf_config_path': 'mlx-community/Phi-3.5-mini-instruct-4bit', 'trust_remote_code': True, 'download_dir': './cache', 'enable_prefix_caching': False, 'gpu_memory_utilization': 0.2, 'disable_log_stats': True, 'enforce_eager': True, 'model': 'mlx-community/Phi-3.5-mini-instruct-4bit'}
WARNING 06-07 17:09:50 [envs.py:2057] Unknown vLLM environment variable detected: VLLM_HOOK_WORKER
WARNING 06-07 17:09:50 [arg_utils.py:1552] The global random seed is set to 0. Since VLLM_ENABLE_V1_MULTIPROCESSING is set to False, this may affect the random state of the Python process that launched vLLM.


INFO 06-07 17:09:51 [model.py:617] Resolved architecture: Phi3ForCausalLM
INFO 06-07 17:09:51 [model.py:1752] Using max model len 4096
WARNING 06-07 17:09:51 [vllm.py:1033] Enforce eager set, disabling torch.compile and CUDAGraphs. This is equivalent to setting -cc.mode=none -cc.cudagraph_mode=none
WARNING 06-07 17:09:51 [vllm.py:1058] Inductor compilation was disabled by user settings, optimizations settings that are only active during inductor compilation will be ignored.
INFO 06-07 17:09:51 [kernel.py:270] Final IR op priority after setting platform defaults: IrOpPriorityConfig(rms_norm=['native'], fused_add_rms_norm=['native'])
INFO 06-07 17:09:51 [platform.py:286] Metal: disabled chunked prefill (non-paged path), max_num_batched_tokens=8192
INFO 06-07 17:09:51 [platform.py:324] Metal memory: 34.4GB total, 10.4GB available
INFO 06-07 17:09:51 [core.py:112] Initializing a V1 LLM engine (v0.22.0) with config: model='mlx-community/Phi-3.5-mini-instruct-4bit', speculative_config=None, 

Rendering prompts:   0%|          | 0/2 [00:00<?, ?it/s]

Processed prompts: 100%|██████████| 2/2 [00:03<00:00,  1.57s/it, est. speed input: 46.49 toks/s, output: 20.38 toks/s]


Finally we can print out the results as follows:

In [12]:
for steered, original in zip(outputs, outputs_original):
    print("=" * 100)
    steered_text = steered.outputs[0].text
    print("\n[With activation steering]\n")
    print(steered_text)
    
    baseline_text = original.outputs[0].text
    print("\n[Without activation steering]\n")
    print(baseline_text)


[With activation steering]

 * The person dressed in a ball gown is wearing a stunning, elaborate outfit that includes a fitted, shimmering dress, high heels

[Without activation steering]

 * The person dressed in a ball gown arrives at the venue, exuding an air of elegance and sophistication, with their

[With activation steering]

 * The 13 colonies were specifically located along the Eastern coast of what is now the United States and were initially established as British territories, later becoming

[Without activation steering]

 * The 13 colonies, also known as the original Thirteen Colonies, were British territories along the eastern coast of North America,


In [13]:
# Minimal-memory smoke test for the steer hook.
# This is designed to help separate hook conversion pressure from overall model pressure.

import gc
from pathlib import Path
from vllm import SamplingParams
from vllm_hook_plugins.metal import HookLLMMetal

try:
    llm.shutdown()
except Exception:
    pass

try:
    del llm
except Exception:
    pass

try:
    import mlx.core as mx
except Exception:
    mx = None

repo_root = Path.cwd().parent.parent if Path.cwd().name == "metal" else Path.cwd()
cache_dir = './cache'
model = 'mlx-community/Phi-3.5-mini-instruct-4bit'
hf_config_path = model
json_path = repo_root / 'model_configs/activation_steer/Phi-3-mini-4k-instruct.json'
dtype = 'auto'
test_cases = [
    "Write a dialogue between two people, one is dressed up in a ball gown and the other is dressed down in sweats. The two are going to a nightly event. Your answer must contain exactly 3 bullet points in the markdown format (use \"* \" to indicate each bullet) such as:\n* This is the first point.\n* This is the second point.",
    "What is the difference between the 13 colonies and the other British colonies in North America? Your answer must contain exactly 6 bullet point in Markdown using the following format:\n* Bullet point one.\n* Bullet point two.\n...\n* Bullet point fix."
]

import json

with open(json_path, "r") as f:
    config = json.load(f)

base_steer = config["steering"]

def _smoke_clear_caches():
    gc.collect()
    if mx is not None and hasattr(mx, "clear_cache"):
        mx.clear_cache()

_smoke_clear_caches()

smoke_llm = HookLLMMetal(
    model=model,
    tokenizer=hf_config_path,
    worker_name="steer_hook_act",
    config_file=json_path,
    download_dir=cache_dir,
    hf_config_path=hf_config_path,
    gpu_memory_utilization=0.35,
    trust_remote_code=True,
    dtype=dtype,
    enable_prefix_caching=False,
    enable_hook=True,
    max_model_len=256,
    max_num_seqs=1,
    max_num_batched_tokens=256,
)

smoke_case = test_cases[0]
smoke_example = smoke_llm.tokenizer.apply_chat_template(
    [{"role": "user", "content": smoke_case}], add_generation_prompt=True, tokenize=False
)
smoke_sampling = SamplingParams(
    temperature=0.0,
    max_tokens=32,
    stop_token_ids=[smoke_llm.tokenizer.eos_token_id, 32007],
    extra_args={"steer": {**base_steer, "method": "add_vector", "coefficient": 10}},
)

print("Running hooked smoke test...")
smoke_hooked = smoke_llm.generate([smoke_example], [smoke_sampling])
print("Hooked smoke test complete")

smoke_llm.llm_engine.reset_prefix_cache()
_smoke_clear_caches()

print("Running baseline smoke test...")
smoke_baseline = smoke_llm.generate([smoke_example], [smoke_sampling], use_hook=False)
print("Baseline smoke test complete")

print("\n[Hooked smoke output]\n")
print(smoke_hooked[0].outputs[0].text)

print("\n[Baseline smoke output]\n")
print(smoke_baseline[0].outputs[0].text)

try:
    smoke_llm.shutdown()
except Exception:
    pass


HookLLMMetal worker=steer_hook_act hooks_enabled=True
INFO 06-07 17:09:55 [utils.py:278] non-default args: {'tokenizer': 'mlx-community/Phi-3.5-mini-instruct-4bit', 'hf_config_path': 'mlx-community/Phi-3.5-mini-instruct-4bit', 'trust_remote_code': True, 'download_dir': './cache', 'max_model_len': 256, 'enable_prefix_caching': False, 'gpu_memory_utilization': 0.35, 'max_num_batched_tokens': 256, 'max_num_seqs': 1, 'disable_log_stats': True, 'enforce_eager': True, 'model': 'mlx-community/Phi-3.5-mini-instruct-4bit'}
WARNING 06-07 17:09:55 [envs.py:2057] Unknown vLLM environment variable detected: VLLM_HOOK_WORKER
WARNING 06-07 17:09:55 [arg_utils.py:1552] The global random seed is set to 0. Since VLLM_ENABLE_V1_MULTIPROCESSING is set to False, this may affect the random state of the Python process that launched vLLM.
INFO 06-07 17:09:55 [model.py:617] Resolved architecture: Phi3ForCausalLM
INFO 06-07 17:09:55 [model.py:1752] Using max model len 256
INFO 06-07 17:09:55 [scheduler.py:239] 

Rendering prompts:   0%|          | 0/1 [00:00<?, ?it/s]

Processed prompts: 100%|██████████| 1/1 [00:01<00:00,  1.25s/it, est. speed input: 65.80 toks/s, output: 25.68 toks/s]

Metal steer: hooked generation complete
Metal steer: cleaning up hook run
Metal steer: host after hook exit rss=2.88 GB available=11.10 GB total=32.00 GB
Hooked smoke test complete
INFO 06-07 17:09:59 [utils.py:278] non-default args: {'tokenizer': 'mlx-community/Phi-3.5-mini-instruct-4bit', 'hf_config_path': 'mlx-community/Phi-3.5-mini-instruct-4bit', 'trust_remote_code': True, 'download_dir': './cache', 'max_model_len': 256, 'enable_prefix_caching': False, 'gpu_memory_utilization': 0.35, 'max_num_batched_tokens': 256, 'max_num_seqs': 1, 'disable_log_stats': True, 'enforce_eager': True, 'model': 'mlx-community/Phi-3.5-mini-instruct-4bit'}
WARNING 06-07 17:09:59 [envs.py:2057] Unknown vLLM environment variable detected: VLLM_HOOK_WORKER
WARNING 06-07 17:09:59 [arg_utils.py:1552] The global random seed is set to 0. Since VLLM_ENABLE_V1_MULTIPROCESSING is set to False, this may affect the random state of the Python process that launched vLLM.


INFO 06-07 17:10:00 [model.py:617] Resolved architecture: Phi3ForCausalLM
INFO 06-07 17:10:00 [model.py:1752] Using max model len 256
WARNING 06-07 17:10:00 [vllm.py:1033] Enforce eager set, disabling torch.compile and CUDAGraphs. This is equivalent to setting -cc.mode=none -cc.cudagraph_mode=none
WARNING 06-07 17:10:00 [vllm.py:1058] Inductor compilation was disabled by user settings, optimizations settings that are only active during inductor compilation will be ignored.
INFO 06-07 17:10:00 [kernel.py:270] Final IR op priority after setting platform defaults: IrOpPriorityConfig(rms_norm=['native'], fused_add_rms_norm=['native'])
INFO 06-07 17:10:00 [platform.py:286] Metal: disabled chunked prefill (non-paged path), max_num_batched_tokens=256
INFO 06-07 17:10:00 [platform.py:324] Metal memory: 34.4GB total, 12.0GB available
INFO 06-07 17:10:01 [core.py:112] Initializing a V1 LLM engine (v0.22.0) with config: model='mlx-community/Phi-3.5-mini-instruct-4bit', speculative_config=None, to

Rendering prompts:   0%|          | 0/1 [00:00<?, ?it/s]

Processed prompts: 100%|██████████| 1/1 [00:01<00:00,  1.29s/it, est. speed input: 63.53 toks/s, output: 24.79 toks/s]

Baseline smoke test complete

[Hooked smoke output]

 * The person dressed in a ball gown is wearing a stunning, elaborate outfit that includes a fitted, shimmering dress, high heels

[Baseline smoke output]

 * The person dressed in a ball gown arrives at the venue, exuding an air of elegance and sophistication, with their
